# 002 Runtime

这是 LangChain Advanced usage 学习线的第二份 Notebook。

官方参考：

- https://docs.langchain.com/oss/python/langchain/runtime

学习目标：

1. 理解 Runtime 是 agent 本轮执行时的运行时对象
2. 区分 `messages`、`state`、`context`、`store`
3. 学会在 tool 中读取 `ToolRuntime.context`
4. 学会用 `ToolRuntime.store` 做跨轮持久记忆
5. 学会用 `ToolRuntime.stream_writer` 发送自定义流事件
6. 学会在 middleware / dynamic prompt 中读取 runtime
7. 对比本仓库 Harness 的 session、ledger、memory、SSE 事件

这一版接入当前项目 `.env` 中的真实模型配置，会真实调用模型。所有工具仍然只做内存读写或进度上报，不执行 shell，也不写业务文件。

## 1. Runtime 的心智模型

Runtime 可以先理解成 agent 执行过程里的“运行时上下文容器”。

用 Java 来类比：

```text
Runtime
  ~= RequestContext + Dependency Injection + ThreadLocal 上下文 + 事件输出通道
```

它不是聊天历史本身，而是让 tools、middleware、prompt 生成器在执行时拿到这些东西：

| 对象 | 作用 | 生命周期 |
| --- | --- | --- |
| `messages` | 当前对话消息 | agent state 的一部分 |
| `state` | agent 当前状态 | 本次图执行期间更新 |
| `context` | 本次调用传入的只读上下文 | 单次 invocation |
| `store` | 跨线程/跨会话的持久存储接口 | 可跨多次 invocation |
| `stream_writer` | 自定义流式事件输出 | 本次 stream 调用 |
| `execution_info` | 当前执行元信息 | 本次执行 |
| `server_info` | MCP/server 相关信息 | 有相关 server 时才有 |

在本仓库 Harness 里，你可以把它对照成：

- `context`：类似 `session_id`、用户信息、请求级配置
- `store`：类似长期记忆或数据库持久层
- `stream_writer`：类似 SSE 的 `answer_delta` / `tool_result` / `approval_required`
- `state`：类似 query loop 的 ledger / completed_steps

## 2. 准备真实模型环境

这一节从项目 `.env` 读取：

- `OPENAI_API_KEY`
- `OPENAI_MODEL`
- `OPENAI_BASE_URL`

如果你使用的是 OpenAI 兼容网关，`OPENAI_BASE_URL` 会被传给 `ChatOpenAI`。

如果后续单元格报 `Connection refused`，通常说明 `.env` 里的模型网关服务没有启动，或者当前机器无法访问该地址。

In [14]:
import os
from dataclasses import dataclass
from pathlib import Path
from typing import Any

from dotenv import load_dotenv
from langchain.agents import create_agent
from langchain.agents.middleware import (
    AgentState,
    ModelRequest,
    before_agent,
    dynamic_prompt,
)
from langchain.tools import ToolRuntime, tool
from langchain_openai import ChatOpenAI
from langgraph.runtime import Runtime
from langgraph.store.memory import InMemoryStore


load_dotenv(Path("../../.env"))
load_dotenv(Path(".env"))

if not os.getenv("OPENAI_API_KEY"):
    raise RuntimeError("未配置 OPENAI_API_KEY，无法运行真实模型版 Runtime 课时。")

real_model = ChatOpenAI(
    model=os.getenv("OPENAI_MODEL") or "gpt-5.4-mini",
    base_url=os.getenv("OPENAI_BASE_URL") or None,
    temperature=0,
)

print("model:", os.getenv("OPENAI_MODEL") or "gpt-5.4-mini")
print("base_url configured:", bool(os.getenv("OPENAI_BASE_URL")))


def print_messages(result: dict) -> None:
    for message in result.get("messages", []):
        print(getattr(message, "type", type(message).__name__), getattr(message, "content", ""))
        tool_calls = getattr(message, "tool_calls", None)
        if tool_calls:
            print("tool_calls:", tool_calls)


model: qwq
base_url configured: True


## 3. 定义 Context Schema

`context` 是你在调用 agent 时传入的请求级上下文。

它适合放：

- 当前用户 ID
- locale / language
- tenant_id
- 请求级权限
- 学习者偏好

它不适合放聊天历史。聊天历史属于 `messages` / `state`。

In [15]:
@dataclass
class UserContext:
    user_id: str
    locale: str
    learner_profile: str


context = UserContext(
    user_id="u_001",
    locale="zh-CN",
    learner_profile="Java 工程师，正在学习 LangChain",
)

context


UserContext(user_id='u_001', locale='zh-CN', learner_profile='Java 工程师，正在学习 LangChain')

## 4. ToolRuntime.context：工具读取请求上下文

工具函数可以声明一个 `runtime: ToolRuntime[UserContext]` 参数。

LangChain 会自动注入 runtime，模型不需要、也不应该自己传这个参数。

这一格会真实调用模型，并要求模型调用 `get_runtime_user` 工具。

In [18]:
@tool
def get_runtime_user(runtime: ToolRuntime[UserContext]) -> str:
    """Return current runtime user info."""
    return (
        "user_id=" + runtime.context.user_id
        + ", locale=" + runtime.context.locale
        + ", learner_profile=" + runtime.context.learner_profile
    )


context_agent = create_agent(
    model=real_model,
    tools=[get_runtime_user],
    context_schema=UserContext,
    system_prompt="你是 Runtime 教学助手。用户询问身份或上下文时，必须先调用 get_runtime_user 工具，再用中文简短回答。",
)

context_result = context_agent.invoke(
    {"messages": [{"role": "user", "content": "请读取 runtime context，告诉我当前用户信息。"}]},
    context=context,
)

print_messages(context_result)


human 请读取 runtime context，告诉我当前用户信息。
ai 
tool_calls: [{'name': 'get_runtime_user', 'args': {}, 'id': 'chatcmpl-tool-bdb8dcc427bb77ee', 'type': 'tool_call'}]
tool user_id=u_001, locale=zh-CN, learner_profile=Java 工程师，正在学习 LangChain
ai 当前用户信息如下：

- **用户ID**: u_001
- **语言环境**: zh-CN
- **学习者档案**: Java 工程师，正在学习 LangChain


## 5. Runtime.store：跨轮持久记忆

`context` 是单次 invocation 的输入。

`store` 是跨 invocation 的存储接口。

区别：

```text
context: 这次请求带进来的上下文
store: 多次请求之间可以保留的数据
```

下面用 `InMemoryStore` 演示保存和读取用户偏好。生产系统里可以换成更可靠的持久化实现。

In [19]:
@tool
def remember_preference(key: str, value: str, runtime: ToolRuntime[UserContext]) -> str:
    """Save a user preference."""
    runtime.store.put(
        ("users", runtime.context.user_id, "preferences"),
        key,
        {"value": value},
    )
    return "saved: " + key + "=" + value


@tool
def read_preference(key: str, runtime: ToolRuntime[UserContext]) -> str:
    """Read a user preference."""
    item = runtime.store.get(("users", runtime.context.user_id, "preferences"), key)
    return item.value["value"] if item else "not found"


store = InMemoryStore()

memory_agent = create_agent(
    model=real_model,
    tools=[remember_preference, read_preference],
    context_schema=UserContext,
    store=store,
    system_prompt=(
        "你是 Runtime 记忆演示助手。"
        "当用户要求保存偏好时，必须调用 remember_preference。"
        "当用户要求读取偏好时，必须调用 read_preference。"
        "工具返回后，用中文简短总结。"
    ),
)

remember_result = memory_agent.invoke(
    {
        "messages": [
            {
                "role": "user",
                "content": "请调用工具保存偏好：key=teaching_style，value=多用 Java 类比。",
            }
        ]
    },
    context=context,
)

print_messages(remember_result)
print("store direct read:", store.get(("users", "u_001", "preferences"), "teaching_style").value)


human 请调用工具保存偏好：key=teaching_style，value=多用 Java 类比。
ai 
tool_calls: [{'name': 'remember_preference', 'args': {'key': 'teaching_style', 'value': '多用 Java 类比'}, 'id': 'chatcmpl-tool-88cb8b6321fb0687', 'type': 'tool_call'}]
tool saved: teaching_style=多用 Java 类比
ai 已保存偏好：teaching_style=多用 Java 类比。
store direct read: {'value': '多用 Java 类比'}


In [20]:
read_result = memory_agent.invoke(
    {"messages": [{"role": "user", "content": "请调用工具读取 key=teaching_style 的偏好。"}]},
    context=context,
)

print_messages(read_result)


human 请调用工具读取 key=teaching_style 的偏好。
ai 
tool_calls: [{'name': 'read_preference', 'args': {'key': 'teaching_style'}, 'id': 'chatcmpl-tool-9ea9c24d5866dd79', 'type': 'tool_call'}]
tool 多用 Java 类比
ai 用户偏好的教学风格是：多用 Java 类比。


## 6. Runtime.stream_writer：工具发自定义流事件

`stream_writer` 允许工具在执行过程中发出自定义流式事件。

这和本仓库 SSE 很像：

- 工具开始执行
- 工具处理中
- 工具执行完成
- 需要审批
- 发生错误

下面用 `stream_mode="custom"` 接收工具发出的自定义进度事件。

In [21]:
@tool
def report_progress(runtime: ToolRuntime[UserContext]) -> str:
    """Report progress with runtime stream writer."""
    runtime.stream_writer({"event": "progress", "message": "开始执行工具"})
    runtime.stream_writer({"event": "progress", "message": "工具执行完成"})
    return "done"


progress_agent = create_agent(
    model=real_model,
    tools=[report_progress],
    context_schema=UserContext,
    system_prompt="你是 Runtime 流式演示助手。用户要求汇报进度时，必须调用 report_progress 工具。",
)

for chunk in progress_agent.stream(
    {"messages": [{"role": "user", "content": "请调用工具并汇报进度。"}]},
    context=context,
    stream_mode="custom",
):
    print(chunk)


{'event': 'progress', 'message': '开始执行工具'}
{'event': 'progress', 'message': '工具执行完成'}


## 7. Dynamic Prompt 读取 Runtime

Runtime 不只给 tool 用，也可以给 middleware 用。

`dynamic_prompt` 可以根据 `runtime.context` 动态生成 system prompt。

例如：同一个 agent 面对 Java 工程师和 Python 初学者，提示词可以不同。

In [22]:
@dynamic_prompt
def prompt_by_runtime(request: ModelRequest) -> str:
    profile = request.runtime.context.learner_profile
    print("dynamic prompt sees profile:", profile)
    return "你是教学助手。请根据学习者背景讲解。学习者背景：" + profile


prompt_agent = create_agent(
    model=real_model,
    tools=[],
    context_schema=UserContext,
    middleware=[prompt_by_runtime],
)

prompt_result = prompt_agent.invoke(
    {"messages": [{"role": "user", "content": "用两句话解释 LangChain Runtime。"}]},
    context=context,
)

print(prompt_result["messages"][-1].content)


dynamic prompt sees profile: Java 工程师，正在学习 LangChain
LangChain Runtime 是 LangGraph 的核心执行引擎，负责管理智能体（Agent）的状态、循环逻辑以及工具调用的生命周期。它通过定义状态图（State Graph）和节点间的转换规则，让开发者能够以声明式的方式构建和运行复杂的多步 AI 工作流。


## 8. Middleware 读取 Runtime

普通 middleware hook 也能读取 `runtime.context`、`runtime.store`。

这适合做：

- 按 tenant / user 做审计
- 按 locale 选择输出策略
- 按用户风险等级决定是否审批
- 把运行信息写入 ledger

In [23]:
@before_agent
def audit_runtime_context(state: AgentState, runtime: Runtime[UserContext]) -> dict[str, Any] | None:
    print("audit user_id:", runtime.context.user_id)
    print("audit locale:", runtime.context.locale)
    print("message_count:", len(state.get("messages", [])))
    return None


audit_agent = create_agent(
    model=real_model,
    tools=[],
    context_schema=UserContext,
    middleware=[audit_runtime_context],
    system_prompt="你是 Runtime 审计演示助手。请用一句中文回答。",
)

audit_result = audit_agent.invoke(
    {"messages": [{"role": "user", "content": "测试 runtime audit"}]},
    context=context,
)

print(audit_result["messages"][-1].content)


audit user_id: u_001
audit locale: zh-CN
message_count: 1
Runtime 审计演示助手已就绪，请提供具体的审计场景或日志数据以便进行测试。


## 9. execution_info / server_info

`ToolRuntime` 还包含：

- `execution_info`：当前工具执行相关元信息
- `server_info`：MCP / server 相关元信息

很多普通本地工具场景下，`server_info` 可能为空。学习时先知道它们属于运行时元数据，不要把业务状态塞进去。

In [24]:
@tool
def inspect_runtime_meta(runtime: ToolRuntime[UserContext]) -> str:
    """Inspect runtime metadata."""
    return (
        "tool_call_id=" + str(runtime.tool_call_id)
        + ", execution_info=" + type(runtime.execution_info).__name__
        + ", server_info=" + type(runtime.server_info).__name__
    )


meta_agent = create_agent(
    model=real_model,
    tools=[inspect_runtime_meta],
    context_schema=UserContext,
    system_prompt="你是 Runtime 元信息演示助手。必须调用 inspect_runtime_meta 工具，再用中文简短总结。",
)

meta_result = meta_agent.invoke(
    {"messages": [{"role": "user", "content": "请检查 runtime 元信息。"}]},
    context=context,
)

print_messages(meta_result)


human 请检查 runtime 元信息。
ai 
tool_calls: [{'name': 'inspect_runtime_meta', 'args': {}, 'id': 'chatcmpl-tool-8683c85bc9aa193c', 'type': 'tool_call'}]
tool tool_call_id=chatcmpl-tool-8683c85bc9aa193c, execution_info=ExecutionInfo, server_info=NoneType
ai Runtime 元信息检查完成。当前执行信息为 `ExecutionInfo`，服务器信息未提供（`NoneType`）。


## 10. Runtime 和 State 的区别

这两个概念很容易混。

| 概念 | 你可以怎么理解 | 典型内容 |
| --- | --- | --- |
| `state` | agent 图运行中的可变工作区 | messages、结构化响应、中间步骤 |
| `context` | 本次调用传入的外部上下文 | user_id、locale、tenant、权限 |
| `store` | 跨调用的持久存储接口 | 用户偏好、长期记忆、业务记录 |
| `runtime` | 把 context/store/stream_writer/meta 打包后注入给执行节点 | 工具和 middleware 的运行时依赖 |

Java 类比：

```text
state   ~= 本次流程内部的工作对象
context ~= Controller 入口传入的请求上下文
store   ~= Repository / Cache / Memory Service
runtime ~= 把这些对象注入给拦截器、工具、服务方法的容器
```

## 11. 和本仓库 Harness 的对应关系

| LangChain Runtime | 本仓库 Harness 可以类比为 | 说明 |
| --- | --- | --- |
| `context` | session/request 级配置 | 当前用户、会话、权限、学习偏好 |
| `store` | 数据库/长期记忆/缓存 | 跨会话保存信息 |
| `stream_writer` | SSE 事件输出 | 进度、工具结果、审批请求 |
| `state` | ledger / completed_steps | 当前 query loop 的中间状态 |
| `ToolRuntime` | 工具执行上下文 | 工具知道是谁在调用、能写哪里、如何上报事件 |

关键判断：

```text
Runtime 不是用来替代业务边界的。
Runtime 是让业务边界能被 tools 和 middleware 正确读取。
```

## 12. 本讲练习

请判断下面信息应该放在哪里：

1. 当前用户 ID
2. 当前这轮对话的 messages
3. 用户长期偏好：喜欢 Java 类比
4. 工具执行过程中的进度事件
5. 本次 agent 的中间步骤记录

参考答案：

1. `context`
2. `state["messages"]`
3. `store`
4. `stream_writer`
5. `state`，本仓库里可类比 ledger / completed_steps

## 13. 本讲小结

这一讲的核心不是记住 `ToolRuntime` 的所有字段，而是理解：

```text
Runtime 是 agent 执行时给 tool 和 middleware 的依赖注入入口。
```

你现在应该能判断：

- 用户信息和请求级配置放 `context`
- 长期记忆放 `store`
- 流式进度放 `stream_writer`
- 当前运行过程放 `state`
- tool 和 middleware 通过 runtime 读取这些对象

下一步可以继续学习 LangChain 的 streaming，或者把 Runtime 思路映射回本仓库的 SSE + approval resume 流程。